# Phase 2: LPIPS + VAE Fine-tune + Iso-budget Sweep

This notebook runs three steps:
- **Step 1 — VAE Fine-tune**: Adapts the SD-VAE encoder+decoder to DETRAC surveillance frames, reducing reconstruction error.
- **Step 2 — Iso-budget Sweep**: Runs the latent attack and PGD at matched budgets on 30 frames, recording DFR and masked LPIPS.
- **Step 3 — Pareto Curve**: Plots DFR vs masked LPIPS (f14) to compare attack effectiveness and imperceptibility.

**Estimated runtime:** ~6 h total, ~12 Colab Pro compute units  
**Prerequisites:** Colab Pro, L4 GPU, Google Drive with project repo


In [ ]:
# Step 1: Clone your repo from GitHub
!git clone https://github.com/03aziz03/pfe-latent-attack.git /content/pfe-latent-attack
%cd /content/pfe-latent-attack
import os; print('Working directory:', os.getcwd())

In [ ]:
# Mount Google Drive and copy large files
# Upload these to Drive once before running:
#
#   My Drive/pfe_assets/
#       best.pt                  <- your YOLOv8 weights (6 MB)
#       images_50/               <- 50 attack eval frames
#       finetune_seqs/           <- 3 DETRAC sequences for VAE fine-tuning
#           MVI_XXXXX/           <- 60 frames each
#           MVI_YYYYY/
#           MVI_ZZZZZ/

from google.colab import drive
drive.mount('/content/drive')

import os, shutil
ASSETS = '/content/drive/MyDrive/pfe_assets'

# Model weights
os.makedirs('runs/yolov8n_detrac', exist_ok=True)
shutil.copy(f'{ASSETS}/best.pt', 'runs/yolov8n_detrac/best.pt')
print('✓ Copied best.pt')

# Attack eval frames
if not os.path.exists('data/images_50'):
    shutil.copytree(f'{ASSETS}/images_50', 'data/images_50')
print(f'✓ data/images_50: {len(os.listdir("data/images_50"))} frames')

# VAE fine-tune sequences
if not os.path.exists('data/finetune_seqs'):
    shutil.copytree(f'{ASSETS}/finetune_seqs', 'data/finetune_seqs')
n_ft = sum(len(os.listdir(f'data/finetune_seqs/{d}'))
           for d in os.listdir('data/finetune_seqs')
           if os.path.isdir(f'data/finetune_seqs/{d}'))
print(f'✓ data/finetune_seqs: {n_ft} frames across '
      f'{len(os.listdir("data/finetune_seqs"))} sequences')

os.makedirs('results/figures/png', exist_ok=True)
os.makedirs('results/figures/pdf', exist_ok=True)
print('✓ Output directories ready')

In [ ]:
# Step 3: Install dependencies
!pip install lpips>=0.1.4 -q
!pip install -r requirements.txt -q
# Verify GPU
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')

## Step 1: VAE Fine-tune (~2h, ~8 units)

Fine-tunes the SD-VAE encoder+decoder on 50 DETRAC frames with a mixed MSE+LPIPS loss.
Saves checkpoint to `runs/vae_detrac/vae_ft.pt`.


In [ ]:
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

!python scripts/finetune_vae.py \
    --data data/finetune_seqs \
    --output runs/vae_detrac \
    --epochs 20 \
    --lr 1e-5 \
    --batch_size 2 \
    --val_frac 0.15 \
    --patience 3
# Decoder-only fine-tuning + bfloat16 mixed precision -- fits in 22 GB
# If still OOM: change --batch_size 2 to --batch_size 1

In [ ]:
import json, matplotlib.pyplot as plt
meta = json.load(open('runs/vae_detrac/ft_meta.json'))
plt.figure(figsize=(8, 4))
plt.plot(meta['per_epoch_train_loss'], label='train')
plt.plot(meta['per_epoch_val_loss'],   label='val')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title(f'VAE fine-tune  |  best val={meta["best_val_loss"]:.6f}')
plt.legend(); plt.tight_layout(); plt.show()
print(f'Epochs run   : {meta["num_epochs_run"]}')
print(f'Train frames : {meta["n_train_frames"]}  '
      f'Val frames: {meta["n_val_frames"]}')
print(f'Best val loss: {meta["best_val_loss"]:.6f}')

## Step 2: Iso-budget sweep (~4h, ~4 units)

Sweeps latent eps ∈ {0.25, 0.50, 1.00} and PGD eps ∈ {4/255, 8/255, 12/255}
on the first 30 frames. Records DFR and masked LPIPS for each config.
Use `--resume` to restart after a Colab disconnect.


In [ ]:
!python scripts/run_iso_budget.py \
    --config configs/phase2.yaml \
    --data data/images_50 \
    --n_frames 30 \
    --output results/iso_budget \
    --resume
# --resume skips any config whose JSON already exists (safe to re-run after disconnect)

In [ ]:
import json, pandas as pd
summary = json.load(open("results/iso_budget/summary.json"))
df = pd.DataFrame(summary).T
print(df[["attack", "eps", "mean_dfr_strict_proportional", "mean_lpips", "n_frames"]].to_string())

## Step 3: Generate Pareto curve

Produces `results/figures/png/f14_pareto_dfr_lpips.png` and the matching PDF.


In [ ]:
!python scripts/generate_pareto.py
from IPython.display import Image
Image("results/figures/png/f14_pareto_dfr_lpips.png")

## Done — save results back to Drive

Copy the key outputs back to Drive so they persist after the Colab session ends.

In [ ]:
import os, shutil
ASSETS = '/content/drive/MyDrive/pfe_assets'

# Save fine-tuned VAE weights back to Drive
shutil.copy('runs/vae_detrac/vae_ft.pt', f'{ASSETS}/vae_ft.pt')
shutil.copy('runs/vae_detrac/ft_meta.json', f'{ASSETS}/ft_meta.json')

# Save iso-budget results back to Drive
os.makedirs(f'{ASSETS}/iso_budget', exist_ok=True)
for f in os.listdir('results/iso_budget'):
    shutil.copy(f'results/iso_budget/{f}', f'{ASSETS}/iso_budget/{f}')

# Save Pareto figure
shutil.copy('results/figures/png/f14_pareto_dfr_lpips.png', f'{ASSETS}/f14_pareto_dfr_lpips.png')

print('Results saved to Drive. Verifying...')
for f in ['runs/vae_detrac/vae_ft.pt',
          'results/iso_budget/summary.json',
          'results/figures/png/f14_pareto_dfr_lpips.png']:
    status = '✓' if os.path.exists(f) else '✗ MISSING'
    print(f'{status}  {f}')